# MergeKit-Paper-Repro: Walkthrough & Azure ML Guide

This notebook walks through reproducing the flagship case study from the
original MergeKit paper (Goddard et al. 2024, [arXiv 2403.13257](https://arxiv.org/abs/2403.13257),
Table 1): merging **Llama-2-7B-Chat** + **Meditron-7B** (a medical-domain
fine-tune) via four methods — **LERP, SLERP, TIES, DARE-TIES** — and
evaluating all of it plus both source checkpoints on six benchmarks (USMLE,
MedMCQA, PubMedQA, ARC-Challenge, HellaSwag, MMLU), directly comparable to
the paper's own published numbers.

It also documents every real bug hit along the way — several took hours of
debugging, and the fixes aren't obvious from the config files alone.

Repo: `alan-turing-institute/model-merging`, folder `merge-job-mergekit-paper-repro/`.


## Prerequisites

1. **Azure CLI with the `ml` extension**, logged into an account with a role
   on the `TIRE-1` resource group / `TIRE-2` workspace. See
   `azure-ml-handover.md` in the repo if you don't have this set up yet.
2. **A Hugging Face access token** (Settings → Access Tokens → New token,
   **Read** scope is enough).
3. **Access to the gated `epfl-llm/meditron-7b` repo** — visit its model
   page on huggingface.co while logged into the account that owns your
   token, and accept its terms. Without this, every job that touches
   Meditron-7B fails with a 401, no matter how valid the token otherwise is.


In [ ]:
import os

# Jupyter kernels don't inherit shell customizations from ~/.zshrc/~/.bash_profile,
# so `!az` can fail with "command not found" even if it works fine in a terminal.
# az is installed in a venv here rather than on the global PATH -- add it once,
# for this kernel session, so every `!az ...` cell below just works.
AZ_VENV_BIN = "/Users/mpietrzyk/azure-cli-venv/bin"
if AZ_VENV_BIN not in os.environ["PATH"]:
    os.environ["PATH"] = AZ_VENV_BIN + os.pathsep + os.environ["PATH"]

!az version

In [ ]:
# Confirm az is set up and pointed at the right workspace.
# (Adjust the path if az isn't already on your PATH.)
!az account show --query name -o tsv
!az ml compute list --resource-group TIRE-1 --workspace-name TIRE-2 -o table


## Repo structure: three kinds of jobs, not two

It's tempting to assume a clean split of "one merge job, one eval job" —
that's not quite how this folder is organized. There are actually three
categories:

| Category | Files | What it does |
|---|---|---|
| **Combined merge+eval** | `job.yml` + `run_all.sh`, `job-dare-ties-fixed.yml` + `run_dare_ties_fixed.sh` | Merges, then evaluates, in one job/container |
| **Eval-only** | `job-eval-only.yml`, `job-eval-merged-only.yml`, `job-eval-lerp-fixed.yml`, `job-eval-single-*.yml` | Never calls `mergekit-yaml` — mounts already-merged data assets (`ro_mount`) or pulls base checkpoints directly from HF Hub |
| **Maintenance** | `job-patch-lerp-config.yml` + `patch_config.sh` | Downloads a merge, patches a config bug, re-uploads as a new data asset version |

There is no standalone "merge-only" job that skips evaluation — merging
always happens bundled with an eval step. The eval-only jobs exist
specifically so a later bug fix or benchmark expansion doesn't force
re-running an expensive merge.


## Option A — combined merge+eval (the canonical one-shot run)

`job.yml` runs `run_all.sh`, which merges all 4 methods, then evaluates all
6 targets (2 base checkpoints + 4 merges) in the same container. This is
the simplest way to run the whole thing, but see the disk-usage gotcha
further down before relying on it for a fresh environment.


In [ ]:
# Illustrative only -- do not run this cell blindly, it submits real
# compute. Replace REPLACE_WITH_YOUR_TOKEN with your own token before running.
#
# This notebook lives inside merge-job-mergekit-paper-repro/ itself, so
# job.yml is already a sibling file in the same directory -- no `cd` needed.
#
# WARNING: don't use angle brackets like <your-token> as a placeholder and
# then forget to replace them -- `<` and `>` are shell redirection
# characters. If that literal text ends up in the command, bash tries to
# parse it as an I/O redirect and crashes with a syntax error before
# anything runs. REPLACE_WITH_YOUR_TOKEN below is plain text specifically
# to avoid that trap.
#
# import os
# os.environ["HF_TOKEN"] = "REPLACE_WITH_YOUR_TOKEN"
# !az ml job create -f job.yml --resource-group TIRE-1 --workspace-name TIRE-2 \
#     --set inputs.hf_token="$HF_TOKEN"
print("See the commented command above -- uncomment and fill in your token to run.")

## Gotcha #1 — `HF_TOKEN` silently not reaching the container

The first fix for removing a hardcoded HF token from these configs used
this pattern:

```yaml
environment_variables:
  HF_TOKEN: ${{inputs.hf_token}}
```

This **looks** correct and validates fine, but the `${{ }}` substitution
does not reliably resolve inside `environment_variables:` — six consecutive
job submissions failed with an identical 401 "gated repo" error on
`epfl-llm/meditron-7b`, even after confirming via a direct curl check that
the token itself was valid every time. The container's `HF_TOKEN` was
literally the unresolved string `${{inputs.hf_token}}`.

**The fix** — move the substitution into the `command:` string instead (the
proven-working context, since it's how every `uri_folder` data input gets
passed elsewhere in this repo), and export it from a positional argument
inside the script:

```yaml
command: >-
  bash run_all.sh ${{outputs.results}} ${{outputs.merged_models}} ${{inputs.hf_token}}
```

```bash
# inside run_all.sh
export HF_TOKEN="$3"
```


## Gotcha #2 — verifying gated access the *right* way

It's tempting to sanity-check a token like this:

```bash
curl -H "Authorization: Bearer $HF_TOKEN" https://huggingface.co/api/models/epfl-llm/meditron-7b
```

**This returns `200` for anyone, authenticated or not** — it's just public
metadata and never actually checks gated access. A literal placeholder
string (copy-pasted from an example command without being replaced) passed
this check multiple times, then failed with a 401 once actually submitted.

The endpoint that actually matters is the file-resolve one MergeKit/
`transformers` really hits:


In [ ]:
import subprocess

def check_meditron_access(token):
    """Returns True if `token` genuinely has access to the gated
    epfl-llm/meditron-7b repo -- checks the real resolve endpoint, not the
    metadata endpoint that returns 200 regardless of auth."""
    result = subprocess.run(
        ["curl", "-s", "-o", "/dev/null", "-w", "%{http_code}",
         "-H", f"Authorization: Bearer {token}",
         "https://huggingface.co/epfl-llm/meditron-7b/resolve/main/config.json"],
        capture_output=True, text=True,
    )
    status = result.stdout.strip()
    print(f"HTTP status: {status}")
    return status == "200"

# check_meditron_access(os.environ["HF_TOKEN"])  # uncomment to actually check
print("Uncomment the call above with your own HF_TOKEN set as an env var.")


## Option B — split-job recovery (parallel, disk-safe)

The combined job merges all 4 methods, keeping every merged 7B output
resident on local disk (~52GB) *before* evaluation even starts loading a
5th model on top. On a fresh environment this can exhaust the container's
disk (a 63GB `/tmp` volume filled to 86% right as the first eval began).

The fix: split each target into its own job. Merged models mount read-only
(`ro_mount`) from a registered data asset instead of sharing local disk with
three other checkpoints; base checkpoints pull one at a time. No single job
ever holds more than ~13-14GB locally. This is also the same pattern
`job-eval-only.yml`/`job-eval-merged-only.yml` already used for recovery
after earlier crashes — just taken one step further, to one job per target
instead of one job for all targets.


In [ ]:
# Illustrative only -- 5 of 6 targets need no HF token and can run in
# parallel immediately; the meditron-7b base eval needs a real token.
#
# import subprocess
# targets = [
#     "job-eval-single-lerp.yml", "job-eval-single-slerp.yml",
#     "job-eval-single-ties.yml", "job-eval-single-dare-ties.yml",
#     "job-eval-single-llama2.yml",
# ]
# procs = [
#     subprocess.Popen(["az", "ml", "job", "create", "-f", f,
#                        "--resource-group", "TIRE-1", "--workspace-name", "TIRE-2"])
#     for f in targets
# ]
# for p in procs:
#     p.wait()
print("See the commented block above for the parallel-submission pattern.")


## Gotcha #3 — the vocab-mismatch bug reappears on every fresh merge

Llama-2-7B-Chat has a 32,000-token vocabulary; Meditron-7B (continued
pretrained for medical text) has 32,017. LERP's merge output can end up
with a `config.json` that declares the *wrong* vocab_size (Meditron's,
32017) while the actual saved tensors are Llama's shape (32000, 4096) —
a MergeKit metadata bug, not real data corruption.

**The tempting workaround, `ignore_mismatched_sizes=True`, is lossy**: it
discards and randomly reinitializes the *entire* mismatched tensor rather
than patching only the ~17 actually-wrong rows. Using it on a fresh LERP
merge produced near-random scores across every benchmark (MMLU dropped to
22.99%, matching this exact bug's signature).

**The real fix** is to patch `config.json`'s `vocab_size` directly. Since
this fresh merge only exists as a data asset (no direct blob RBAC
available), the patch itself runs as a small job: download the merge,
fix the one field, write the corrected copy to an output, then register
that output as a new data asset version.


In [ ]:
# patch_config.sh -- the actual fix, run inside a job with the merge
# mounted via `download` input + `rw_mount` output (rw_mount is NOT valid
# for job *inputs*, only outputs -- that's why this copies rather than
# patching in place).
patch_script = '''
#!/bin/bash
set -euo pipefail
SRC="$1"
DST="$2"

cp -r "$SRC"/. "$DST"/

python3 -c "
import json
p = '$DST/config.json'
d = json.load(open(p))
print('before:', d.get('vocab_size'))
d['vocab_size'] = 32000
json.dump(d, open(p, 'w'), indent=2)
print('after:', d['vocab_size'])
"
'''
print(patch_script)

# After the patch job completes, register its output as a new version:
#
# !az ml data create --name mergekit-repro-lerp --type uri_folder \
#     --path azureml://datastores/workspaceblobstore/paths/azureml/<patch-job-name>/patched_model \
#     --resource-group TIRE-1 --workspace-name TIRE-2
#
# Any job referencing azureml:mergekit-repro-lerp@latest automatically picks
# up the new version on its next submission -- no other config changes needed.


## Monitoring and downloading results

```bash
az ml job show --name <job-name> --resource-group TIRE-1 --workspace-name TIRE-2 --query status -o tsv
az ml job stream --name <job-name> --resource-group TIRE-1 --workspace-name TIRE-2
az ml job download --name <job-name> --resource-group TIRE-1 --workspace-name TIRE-2 -o results
```


In [ ]:
import json
import glob

PAPER = {
    "llama2-7b-chat": {"medqa_4options": 35.90, "medmcqa": 35.45, "pubmedqa": 73.40, "arc_challenge": 44.20, "hellaswag": 55.40, "mmlu": 46.37},
    "meditron-7b":     {"medqa_4options": 38.40, "medmcqa": 24.07, "pubmedqa": 71.40, "arc_challenge": 40.20, "hellaswag": 54.50, "mmlu": 33.06},
    "merged-lerp":      {"medqa_4options": 39.10, "medmcqa": 36.65, "pubmedqa": 75.60, "arc_challenge": 46.76, "hellaswag": 58.66, "mmlu": 48.44},
    "merged-slerp":     {"medqa_4options": 39.20, "medmcqa": 36.91, "pubmedqa": 75.60, "arc_challenge": 46.84, "hellaswag": 58.67, "mmlu": 47.97},
    "merged-ties":      {"medqa_4options": 38.73, "medmcqa": 32.27, "pubmedqa": 75.60, "arc_challenge": 45.05, "hellaswag": 58.23, "mmlu": 45.03},
    "merged-dare-ties": {"medqa_4options": 36.37, "medmcqa": 27.56, "pubmedqa": 72.20, "arc_challenge": 42.92, "hellaswag": 54.79, "mmlu": 41.17},
}
METRIC = {"medqa_4options": "acc", "medmcqa": "acc", "pubmedqa": "acc",
          "arc_challenge": "acc_norm", "hellaswag": "acc_norm", "mmlu": "acc"}

def load_and_compare(results_dir, model_key):
    """Load a downloaded job's results_*.json and compare against PAPER[model_key]."""
    files = glob.glob(f"{results_dir}/**/results_*.json", recursive=True)
    if not files:
        raise FileNotFoundError(f"No results_*.json under {results_dir}")
    data = json.load(open(sorted(files)[-1]))
    paper = PAPER[model_key]
    diffs = []
    header_task, header_ours, header_paper, header_diff = "Task", "Ours", "Paper", "Diff"
    print(f"{header_task:15s} {header_ours:>8s} {header_paper:>8s} {header_diff:>8s}")
    for task, paper_val in paper.items():
        r = data["results"].get(task, {})
        m = METRIC[task]
        val = r.get(f"{m},none", r.get(m))
        if val is None:
            print(f"{task:15s} {'MISSING':>8s}")
            continue
        val *= 100
        diff = val - paper_val
        if task != "hellaswag":  # known model-independent benchmark-config offset
            diffs.append(abs(diff))
        print(f"{task:15s} {val:8.2f} {paper_val:8.2f} {diff:+8.2f}")
    print(f"\nmean |diff| (excl hellaswag): {sum(diffs)/len(diffs):.2f}")

# Example:
# load_and_compare("results/lm-eval/merged-lerp", "merged-lerp")
print("Call load_and_compare(path, model_key) once you have real results downloaded.")


## Results from this reproduction (real data)

A full split-job run against all 6 targets, compared to a prior independent
run of the same experiment three weeks earlier:

| Target | This run | Prior run | Delta |
|---|---|---|---|
| llama2-base | 1.22 | 1.23 | +0.01 (rounding only) |
| meditron-base | 3.36 | 3.37 | +0.01 (rounding only) |
| LERP | 1.34 | 1.34 | +0.00 |
| SLERP | 1.39 | 1.39 | +0.00 |
| TIES | 1.35 | 1.35 | +0.00 |
| DARE-TIES | 1.99 | 2.88 | −0.89 |

Five of six targets are essentially perfectly reproducible across two
independent runs — expected, since none of them involve any randomness in
either the merge or the loglikelihood-based eval. DARE-TIES is the one
genuine source of run-to-run variance, and it's fully explained rather than
mysterious: MergeKit's DARE dropout mask (`torch.bernoulli`) is unseeded by
default, so every DARE-TIES run is one noisy draw from the same
distribution, not a fixed, reproducible number.

Per-benchmark breakdown (raw scores, this run):

| Benchmark | llama2-base | meditron-base | LERP | SLERP | TIES | DARE-TIES |
|---|---|---|---|---|---|---|
| USMLE (medqa) | 39.28 | 27.26 | 41.71 | 41.79 | 35.35 | 33.70 |
| MedMCQA | 37.41 | 26.63 | 40.57 | 40.57 | 34.50 | 29.57 |
| PubMedQA | 73.40 | 70.60 | 75.60 | 75.60 | 76.00 | 73.40 |
| ARC-Challenge | 43.52 | 41.89 | 46.67 | 46.50 | 45.56 | 44.20 |
| HellaSwag | 75.98 | 72.93 | 76.81 | 76.90 | 75.62 | 73.23 |
| MMLU | 46.47 | 32.43 | 48.37 | 48.33 | 45.26 | 38.36 |

Reading across the HellaSwag row: every single target lands +17 to +21
points above the paper, including both unmerged source checkpoints — a
uniform, model-independent offset pointing to a benchmark-configuration
difference (most likely few-shot count) rather than anything about the
merges. Meditron-base's `medqa` result (27.26 vs. paper's 38.40, a genuine
−11.14 outlier) stands alone as the only cell in this whole table beyond
roughly ±4 points.


## Appendix — full bug log for this reproduction

In the order they were hit, across the original run and this session's
independent re-run:

1. **`huggingface_hub>=1.16`** tightened its `hf://` URI parser, breaking
   `datasets`' loader for bare-slug canonical datasets (`medmcqa`) —
   fixed by pinning `huggingface_hub<1.16` in the Dockerfile.
2. **`datasets>=4.0`** removed script-based dataset loaders entirely,
   breaking `pubmedqa` (`bigbio/pubmed_qa` ships a builder script) —
   fixed by pinning `datasets<4.0.0`.
3. **`pubmedqa` still needs `trust_remote_code`** even with the pin above
   — `lm_eval`'s own model-arg doesn't propagate to dataset loading (a
   known open upstream issue). Fixed via the `HF_DATASETS_TRUST_REMOTE_CODE=1`
   env var, `datasets`' own opt-in mechanism. This one was *re-introduced*
   when the split-job scripts were first written (forgot to carry it over)
   and hit all 6 targets again before being fixed a second time.
4. **Vocab-size mismatch** (Llama-2-7B-Chat=32000 vs. Meditron-7B=32017)
   breaks LERP's `config.json` metadata specifically — `ignore_mismatched_sizes=True`
   is a lossy workaround (discards the whole tensor); the real fix patches
   `vocab_size` directly. This bug reappears on *every fresh LERP merge*,
   not just the first time it was found.
5. **`dare_ties`'s default `rescale=True`** can overflow `float16` to NaN
   during merge for vocab-mismatched tensors specifically — fixed via
   `dtype: bfloat16` in the merge config.
6. **`${{inputs.X}}` templating doesn't reliably resolve inside
   `environment_variables:`** — moving the same substitution into the
   `command:` string (a context proven to work for `uri_folder` inputs
   everywhere else in this repo) fixed it.
7. **The wrong HF endpoint for verifying gated access** (`/api/models/{repo}`
   returns 200 regardless of authentication) let an unreplaced placeholder
   token pass a pre-flight check that should have caught it.
8. **Combined merge+eval disk exhaustion** — keeping all 4 merged 7B
   outputs resident locally (~52GB) before evaluation starts loading a 5th
   model exhausts a 63GB container disk. Fixed by splitting into one job
   per target, each mounting read-only instead of sharing local disk.
9. **`rw_mount` is not a valid mode for job *inputs*** (only `download`,
   `ro_mount`, `direct`) — only valid for outputs. The config-patch job had
   to download-then-copy-to-a-writable-output instead of mounting the
   existing asset read-write in place.
